# 1. LIBRARIES

In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path to allow imports from functions_final.py
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib as mpl
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score

# Import functions from functions_final.py
from functions_final import *
from UQpy.distributions import Uniform, JointIndependent

# Set matplotlib parameters for better aesthetics
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False,
                        'figure.dpi': 100
                    })

# 2. LOADING THE CARBONATION MODEL

### 2.1 Load

In [3]:
# Path to the trained model
# name_best_model = r'D:\Documentos\ic_victor\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'
# name_best_model = r'/home/wmpjrufg/Documents/2024-1_victor_hugo_renata_maria/beam_problem_1/model_NeuralNetwork_MLP_fold_4.pkl'
# name_best_model = r'D:\py\2024-1_victor_hugo_renata_maria\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'
name_best_model = r'D:\github\2024-1_victor_hugo_renata_maria\beam_problem_1\model_NeuralNetwork_MLP_fold_4.pkl'

# Load the model
model = joblib.load(name_best_model)
print("Carbonation model loaded successfully!")
print(f"   Expected features: {model.feature_names_in_}")

Carbonation model loaded successfully!
   Expected features: ['CO2 (%)' 'fc (MPa)' 'RH (%)' 'Type of cement' 'Exposure conditions'
 't (years)']


### 2.2 Testing model

In [7]:
mat = {
        'f_ck [MPa]': 30,
        'Type of cement': 2
      }
expo = {
          'Installation year': 1990,
          'Exposure conditions': 2,
          'Relative humidity [%]': 60
        }
geo = {'cover[mm]':30}
load = {}
predictor = CO2Predictor()
beam_with_rh = Beam(geo=geo, mat=mat, load=load, expo=expo)
predictor.set_beam(beam_with_rh)
profile = predictor.carbonation_profile(model_=model, lifetime=150)

### 2.3 Carbonation profile over time

In [8]:
profile

,calendar year,t (years),CO2 (%),carbonation depth (mm)
0,1990,0,0.03574,2.185264
1,2000,10,0.03690,13.843039
2,2010,20,0.03893,19.512542
3,2020,30,0.04132,24.492433
4,2030,40,0.04407,28.402407
5,2040,50,0.04718,31.645972
6,2050,60,0.05065,34.647134
7,2060,70,0.05448,37.518476
8,2070,80,0.05867,40.403024
9,2080,90,0.06322,43.517887


### 2.4 Test carbonation depth at 2025

In [9]:
carb_depth_mm = predictor.carbonation_depth_at_time(profile, 2025)
carb_depth_mm

26.44741990254805

# 3. DEFINITION OF RANDOM VARIABLES

Defines the probability distributions for the input variables:
- Concrete compressive strength ($f_{ck}$);
- Relative humidity (RH).
- Cover depth (cov).

### 3.1 Design variables

In [10]:
fck_min = 20 # MPa
fck_max = 50 # MPa
rh_min  = 50 # %
rh_max  = 80 # %
cov_min = 20  # mm
cov_max = 60  # mm

### 3.2 Fixed parameters for the analysis

In [11]:
cement_type          = 3
installation_year    = 1990
exposure_conditions  = 2
n_samples            = 100        # Number of design samples. Use 1 for testing one sample
n_latent_samples     = 1000       # Number of latent samples per design sample
n_samples_validation = 3          # Number of validation samples
n_lambdas            = 4          # Number of λs to be predicted (λ1, λ2, λ3, λ4)

### 3.3 Samples

In [13]:
# Distributions of random variables
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist = Uniform(loc=cov_min, scale=cov_max - cov_min)

# Joint distribution
joint = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

# Generate samples
x_pce_rvs = joint.rvs(n_samples)

# Report sample statistics
print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations: {n_samples * n_latent_samples}")
print("\nSample statistics:")
print(f"   fck:      {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:       {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover:    {x_pce_rvs[:, 2].min():.1f} - {x_pce_rvs[:, 2].max():.1f} mm (mean: {x_pce_rvs[:, 2].mean():.1f} mm)")

Samples generated successfully!
   Number of design samples: 100
   Number of latent samples per design sample: 1000
   Total simulations: 100000

Sample statistics:
   fck:      20.1 - 49.6 MPa (mean: 33.0 MPa)
   RH:       50.0 - 80.0% (mean: 65.2%)
   cover:    20.0 - 59.6 mm (mean: 41.1 mm)


# 4. EVALUATION OF CARBONATION PROGRESS USING GENERALIZED LAMBDA DISTRIBUTION

### 4.1 Step time

In [ ]:
times               = np.linspace(0, 150, 20, endpoint=True)  # Time points for carbonation depth prediction
# times               = [10, 20, 30, 40]
complete_model_list = []
pce_dataset         = []
times 

array([  0.        ,   7.89473684,  15.78947368,  23.68421053,
        31.57894737,  39.47368421,  47.36842105,  55.26315789,
        63.15789474,  71.05263158,  78.94736842,  86.84210526,
        94.73684211, 102.63157895, 110.52631579, 118.42105263,
       126.31578947, 134.21052632, 142.10526316, 150.        ])

### 4.2 Loop over design samples

In [24]:
print("="*60)
print("BUILDING THE DURABILITY EMULATOR")
print("="*60)

for t in times:
    # ============================================================
    # EMULATOR FUNCTION - CARBONATION DEPTH AND LAMBDAS
    # ============================================================
    print(f'\n{"-"*40}')
    print(f'PROCESSING EMULATOR FOR TIME STEP: {t} years')
    print(f'{"-"*40}')
    df_full, df_unique = emulator_function_time_durability(
                                                                x=x_pce_rvs,
                                                                names_x_variables=["fck", "rh", "cov"],
                                                                carb_model=model,
                                                                cement_type=cement_type,
                                                                installation_year=installation_year,
                                                                exposure_conditions=exposure_conditions,
                                                                time_step=t,
                                                                n_latent_samples=n_latent_samples,
                                                                verbose=False
                                                            )
    # Save the dataset for this time step
    filename = f'dataset_full_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename, 'wb') as f:
        dill.dump(df_full, f)
    filename = f'dataset_unique_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename, 'wb') as f:
        dill.dump(df_unique, f)
    print(f'1. The dataset has been saved!')
    # =============================================================
    # BUILDING THE PCE METAMODEL
    # =============================================================
    lambda_cols      = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']
    y_pce_rvs        = df_unique[lambda_cols].to_numpy()
    max_degree       = 3
    polynomial_basis = TotalDegreeBasis(joint, max_degree)
    least_squares    = LeastSquareRegression()
    pce_metamodel    = PolynomialChaosExpansion(polynomial_basis=polynomial_basis, regression_method=least_squares)                                                                                                        
    # Train
    pce_metamodel.fit(x_pce_rvs, y_pce_rvs)
    # Save the PCE metamodel for this time step
    filename = f'pce_metamodel_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename, 'wb') as f:
        dill.dump(pce_metamodel, f)
    print(f'2. PCE training dataset has been saved!')
    # =============================================================
    # VALIDATION THE PCE METAMODEL
    # =============================================================
    x_pce_rvs_val = joint.rvs(n_samples_validation)
    df_full_val, df_unique_val = emulator_function_time_durability(
                                                                        x=x_pce_rvs_val,
                                                                        names_x_variables=["fck", "rh", "cov"],
                                                                        carb_model=model,
                                                                        cement_type=cement_type,
                                                                        installation_year=installation_year,
                                                                        exposure_conditions=exposure_conditions,
                                                                        time_step=t,
                                                                        n_latent_samples=n_latent_samples,
                                                                        verbose=False
                                                                    )
    lambda_cols      = ['lambda 1', 'lambda 2', 'lambda 3', 'lambda 4']
    y_pce_val_true   = df_unique_val[lambda_cols].to_numpy()
    y_pce_val_pred   = pce_metamodel.predict(x_pce_rvs_val)
    mse_por_lambda   = []
    r2_por_lambda    = []
    for ii in range(n_lambdas):
        verdade = y_pce_val_true[:, ii]
        predito = y_pce_val_pred[:, ii]
        # MSE computing
        mse = mean_squared_error(verdade, predito)
        mse_por_lambda.append(mse)
        # R² computing
        r2 = r2_score(verdade, predito)
        r2_por_lambda.append(r2)
    statistics_ = pd.DataFrame({'MSE λ1': mse_por_lambda[0], 'MSE λ2': mse_por_lambda[1], 'MSE λ3': mse_por_lambda[2], 'MSE λ4': mse_por_lambda[3], 'R² λ1': r2_por_lambda[0], 'R² λ2': r2_por_lambda[1], 'R² λ3': r2_por_lambda[2], 'R² λ4': r2_por_lambda[3]}, index=[0])
    filename_stats = f'pce_validation_stats_{t}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}.pkl'
    with open(filename_stats, 'wb') as f:
        dill.dump(statistics_, f)
    print(f'3. PCE statistcs has been saved!')

BUILDING THE DURABILITY EMULATOR

----------------------------------------
PROCESSING EMULATOR FOR TIME STEP: 0.0 years
----------------------------------------


KeyboardInterrupt: 

In [ ]:
# filename_stats = f'pce_validation_stats_0_install_1990_cement_3_exposure_2.pkl'
# with open(filename_stats, 'rb') as f:
#     file_loaded = dill.load(f)
# file_loaded

,MSE λ1,MSE λ2,MSE λ3,MSE λ4,R² λ1,R² λ2,R² λ3,R² λ4
0,0.03006,0.001142,0.001964,0.000587,0.999459,0.89943,-0.078596,-0.432184
